In [1]:
import time
import random
import pandas as pd 
import requests
from bs4 import BeautifulSoup

In [2]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
}

In [3]:
base_url ='https://www.zameen.com/Plots/Islamabad-3-{}.html'

In [4]:
all_plots = []

In [5]:
START_PAGE = 1
MAX_PAGES = 1290

In [ ]:
for page in range(START_PAGE,MAX_PAGES + 1):
    url = base_url.format(page)
    print(f"Scraping page {page}...")
    print(f"[{page}/ {MAX_PAGES}] Fetching: {url}")

    try:
        response = requests.get(url, headers=headers, timeout = 15)

        if response.status_code == 403:
            print(f"Encountered 403 Forbidden on page {page}. Sleeping 60s before retry...")
            time.sleep(60)
            response = requests.get(url, headers=headers, timeout=15)
            if response.status_code != 200:
                print(f"Persistent failure on page {page} ({response.status_code}). Stopping.")
                break
        elif response.status_code != 200:
            print(f"Unexpected status code {response.status_code} on page {page}. Stopping.")
            break

        soup = BeautifulSoup(response.text, 'lxml')

        cards = soup.find_all('li', role='article')
        if not cards:
            cards = soup.find_all('article')
        for card in cards:
            price_tag = card.find('span', {'aria-label': 'Price'})
            price = price_tag.get_text(strip=True) if price_tag else None

            loc_tag = card.find('div', {'aria-label': 'Location'})
            location = loc_tag.get_text(strip=True) if loc_tag else None

            area_tag = card.find('span', {'aria-label': 'Area'})
            area = area_tag.get_text(strip=True) if area_tag else None

            title_tag = card.find('h2') or card.find('a', title=True)
            title = title_tag.get_text(strip=True) if title_tag else None

            link_tag = card.find('a', href=True)
            link = ("https://www.zameen.com" + link_tag['href']) if link_tag else None

            if price or location:
                all_plots.append({
                    'title': title,
                    'price':price,
                    'location': location,
                    'area': area,
                    'link': link,
                    'page_scraped': page
                })
        if page % 50 == 0:
            checkpoint_df = pd.DataFrame(all_plots)
            checkpoint_df.to_csv('islamabad_plots_checkpoint.csv', index=False)
            print(f">>> Checkpoint saved: {len(checkpoint_df)} records stored.")
        time.sleep(random.uniform(2.0, 4.0))

    except requests.exceptions.RequestException as e:
        print(f"Network error on page {page}: {e}. Halting run.")
        break
    except Exception as e:
        print(f"Unexpected error on page {page}: {e}. Halting run.")
        break
df = pd.DataFrame(all_plots)

if not df.empty:
    # Separate the specific sector/society name from the full location string
    df['sector'] = df['location'].apply(lambda x: x.split(',')[0].strip() if pd.notnull(x) else None)
    
    # Export full dataset
    df.to_csv('zameen_islamabad_plots_full.csv', index=False)
    print(f"\nScraping complete! Total plots saved: {len(df)}")
else:
    print("\nNo data was collected.")

Scraping page 1...
[1/ 1290] Fetching: https://www.zameen.com/Plots/Islamabad-3-1.html
Scraping page 2...
[2/ 1290] Fetching: https://www.zameen.com/Plots/Islamabad-3-2.html
Scraping page 3...
[3/ 1290] Fetching: https://www.zameen.com/Plots/Islamabad-3-3.html
Scraping page 4...
[4/ 1290] Fetching: https://www.zameen.com/Plots/Islamabad-3-4.html
Scraping page 5...
[5/ 1290] Fetching: https://www.zameen.com/Plots/Islamabad-3-5.html
Scraping page 6...
[6/ 1290] Fetching: https://www.zameen.com/Plots/Islamabad-3-6.html
Scraping page 7...
[7/ 1290] Fetching: https://www.zameen.com/Plots/Islamabad-3-7.html
Scraping page 8...
[8/ 1290] Fetching: https://www.zameen.com/Plots/Islamabad-3-8.html
Scraping page 9...
[9/ 1290] Fetching: https://www.zameen.com/Plots/Islamabad-3-9.html
Scraping page 10...
[10/ 1290] Fetching: https://www.zameen.com/Plots/Islamabad-3-10.html
Scraping page 11...
[11/ 1290] Fetching: https://www.zameen.com/Plots/Islamabad-3-11.html
Scraping page 12...
[12/ 1290] Fetchi